<a href="https://colab.research.google.com/github/Ragul2526/RAG-based-Document-QA-Assistant/blob/main/qa_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Load PDF

We use PyMuPDF ('fitz') to open a PDF and extract its text page by page. This gives us a list of Strings - one per page, that we will later chunk and embed.

Upload any pdf to test.

In [ ]:
!pip install pymupdf -q
import fitz
from google.colab import files

print("Upload a pdf file:")
f = files.upload()
f_name = list(f.keys())[0]

print("Loaded ", f_name)

doc = fitz.open(f_name)
pages = [page.get_text() for page in doc]

print(f"Total pages :{len(pages)}")
print("==== Preview of the content ====")
print(pages[0][:500])

##Chunk the text

Pages are too large and uneven to search over directly.
We split all page text into fixed size overlapping chunks.

- chunk_size = 500 chars -> small enough to be focused
- chunk_overlap = 100 chars -> prevent key sentences being cut off at bounderies

Each chunk stores the page it came from so it will be easier to reference later.

In [ ]:
def chunk_text(pages, chunk_size = 500, chunk_overlap = 100):
  chunks = []
  chunk_index = 0
  for page_num, page_text in enumerate(pages):
    page_text = page_text.strip()
    if not page_text:
      continue
    s = 0
    while s < len(page_text):
      e = s + chunk_size
      chunk = page_text[s:e]

      chunks.append(
          {
           "text" : chunk,
           "page_number" : page_num + 1,
           "chunk_index" : chunk_index
          }
      )
      chunk_index += 1
      s += chunk_size - chunk_overlap
  return chunks

chunks = chunk_text(pages)
print("Total chunks created : ",len(chunks), ", from ", len(pages), "pages")

for i in range(3):
  c = chunks[i]
  print("Chunk -",c["chunk_index"],",Page number:",c["page_number"])
  print(c["text"],"\n")
print("-- Overlap check (last 100 chars of chunk 0 vs first 100 of chunk 1) --")
print("END of chunk 0:", repr(chunks[0]['text'][-100:]))
print("START of chunk 1:", repr(chunks[1]['text'][:100]))

##Embed Each Chunks

We convert every chunk's text into a 384-dimensional vector using
the `all-MiniLM-L6-v2` sentence embedding model.

These vectors capture *semantic meaning*, not just keywords.
Later, when a user asks a question, we embed the question the same way
and find whichever chunk vectors are closest - that's retrieval.

In [ ]:
#!pip install sentence_transformers -q
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embed Model loaded!\n")

chunk_texts = [c["text"] for c in chunks]

print("Embedding ",len(chunk_texts),"chunks")
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)

print(f"Done Embedding\n Embedding shape = {embeddings.shape}")

from numpy.linalg import norm
def cosine_sim(a,b):
  return np.dot(a,b)/(norm(a) * norm(b))
